# App 8 · Capstone — 五天的工业栈，怎么收进一个文件夹

读到这里你应该已经走完七节：环境自检、ReAct、RAG、Code Agent、Multi-Agent、MCP、Skills、LLMOps。每一节是一个独立的能力模块，但**生产里没人会单独用其中一个**——真实业务系统是这些能力的合体。

这一节我们看一个具体的合体案例：把 Multi-Agent 的 Planner/Worker/Reviewer 三角、MCP 的工具协议、Agentic RAG 的 Hybrid+Self 检索、LLMOps 的 trace 全栈集成成一个 pipeline，**整个再打包成一个可以 fork 的 Anthropic Skill 文件夹**。

```
                    User Question
                          ↓
              ┌──────────────────────┐
              │   PlannerAgent       │  ← Multi-Agent
              │   决定走哪条路径      │
              └──┬─────────┬───────┬─┘
                 │         │       │
       ┌─────────┘         │       └────────┐
       ▼                   ▼                ▼
┌──────────────┐  ┌─────────────────┐  ┌────────────┐
│ Agentic RAG  │  │ MCP Tool Call   │  │ Direct LLM │  ← MCP, RAG
│ (Hybrid+Self)│  │ (订单/库存)      │  │            │
└──────┬───────┘  └─────────┬───────┘  └─────┬──────┘
       │                    │                │
       └─────────┐    ┌─────┘    ┌───────────┘
                 ▼    ▼    ▼
           ┌──────────────────────┐
           │ ReviewerAgent        │  ← Multi-Agent
           └─────────┬────────────┘
                     ▼
           ┌──────────────────┐
           │ Final Answer     │
           └──────────────────┘

整条 pipeline wrapped in @observe → trace tree    ← LLMOps
最终打包成 Skill 给团队复用                        ← Skills
```

完整可跑代码在 [`assets/enterprise_5days/skills_demo/capstone_assistant/`](../assets/enterprise_5days/skills_demo/capstone_assistant/) ——它是一个真实的 Anthropic Skill 文件夹，含 `SKILL.md` + `pipeline.py` + `eval.py` + `reference/`。**任何团队拿到这个文件夹就能 fork 改造**。这就是 Skill 格式相对于"散落代码 + 文档 + Wiki"的核心价值：**工程整合 = 一个可复制的目录**。

下面我们一键加载这个 Skill，跑 4 类 query 看路由，最后用打包的 batch eval 评测综合表现。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"📂 repo root: {_root}")


📂 repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


## 1. 一键加载 Capstone Skill

Capstone Skill 是一个完整的 Python 模块——`pipeline.py` 暴露 `upgraded_pipeline(query) -> dict` 入口，内部自动起 vector store、注册 MCP server、装好 Multi-Agent 编排器。我们用 `importlib` 显式从 Skill 目录加载它，看到的就是真实的工程整合：**不是从 notebook 自己写一遍 pipeline，而是 import 一个外部 Skill**——这才是 Skill 格式的核心使用方式。

In [2]:
import importlib.util, sys, os
from pathlib import Path

# 自动定位 repo root（无论 CWD 在哪都能跑）
_root = Path(os.path.abspath("")).resolve()
while not (_root / "Applications").is_dir() and not (_root / "utils").is_dir():
    if _root.parent == _root:
        raise RuntimeError("找不到 repo root（应含 Applications/ 与 utils/）")
    _root = _root.parent

# Capstone Skill 的 pipeline 绝对路径
_pipeline_path = _root / "assets" / "enterprise_5days" / "skills_demo" / "capstone_assistant" / "pipeline.py"
if not _pipeline_path.exists():
    raise FileNotFoundError(f"找不到 capstone pipeline: {_pipeline_path}")

# Load
spec = importlib.util.spec_from_file_location("capstone_pipeline", str(_pipeline_path))
capstone = importlib.util.module_from_spec(spec)
print(f"加载 Capstone Skill (会 setup env + 起 vector store + 启 mcp server, 5-10s)...")
print(f"  pipeline: {_pipeline_path}")
spec.loader.exec_module(capstone)
print("✓ 就位")


加载 Capstone Skill (会 setup env + 起 vector store + 启 mcp server, 5-10s)...
  pipeline: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\assets\enterprise_5days\skills_demo\capstone_assistant\pipeline.py
[OK] 已加载配置: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\.env
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus
  Embedding: dashscope / text-embedding-v3


[LLM] dashscope / qwen-plus
[Embedding] dashscope / text-embedding-v3 (dim=1024)


✓ 已添加 7 个文档，总计 7 个
✓ 就位


## 2. 端到端 demo：一个 query 走完四大组件

下面跑 4 类不同的 query，看 Capstone 怎么自动选择路径：

| 查询类型 | 期望路径 | 走到的组件 |
|---|---|---|
| HR 政策（"年假几天"）| `rag` | Agentic RAG → Reviewer |
| 订单查询（"ORD-002 状态"）| `mcp` | MCP tool → Reviewer |
| 技术文档（"API 限流"）| `rag` | Agentic RAG → Reviewer |
| 闲聊（"你好"）| `direct` | LLM 直答 → Reviewer |

**注意 Reviewer 始终在场**——这是 Multi-Agent 模式的体现：每条答案都过 Reviewer 检查一遍，REJECT 的会被标记。看下面的输出你会发现 Reviewer 偶尔会 REJECT RAG 的答案——这是正常的，说明评审机制在工作（不是橡皮图章）。

In [3]:
# 跑 4 类 query 看 Capstone 自动路由
test_queries = [
    "入职 8 年有几天年假？",         # → rag (HR knowledge)
    "查订单 ORD-002 状态",            # → mcp (order tool)
    "API 限流是多少？",                # → rag (tech knowledge)
    "你好",                            # → direct (chitchat)
]

for q in test_queries:
    r = capstone.upgraded_pipeline(q)
    print(f"\nQ: {q}")
    print(f"  路径: {r['path']}")
    print(f"  答 (前 100 字): {r['answer'][:100]}")
    print(f"  Reviewer: {r['review'][:80]}")



Q: 入职 8 年有几天年假？
  路径: rag
  答 (前 100 字): 10 天
  Reviewer: REJECT，未说明依据（如《职工带薪年休假条例》规定工作满1年不满10年享5天，满10年才10天），且答案错误（8年应为5天）。



Q: 查订单 ORD-002 状态
  路径: mcp
  答 (前 100 字): 调 query_order({'order_id': 'ORD-002'}) → {"status": "pending", "total": 89.0, "customer": "bob"}
  Reviewer: APPROVE —— 回答准确调用指定函数并返回结构化、语义完整的订单状态信息，包含关键字段（status、total、customer），符合查询意图且无冗余



Q: API 限流是多少？
  路径: rag
  答 (前 100 字): 免费版 60 请求/分钟，企业版 6000 请求/分钟。
  Reviewer: REJECT，未说明限流是按 API Key、用户账号还是其他维度统计，也未提及是否支持突发流量（如令牌桶/漏桶算法）、是否区分读写请求、是否有全局/单接口限流



Q: 你好
  路径: direct
  答 (前 100 字): 你好！很高兴见到你，有什么可以帮您的吗？😊
  Reviewer: APPROVE —— 回应礼貌、友好、及时，符合基础对话礼仪，且以开放性提问（“有什么可以帮您的吗？”）自然引导后续交流，表情符号使用得当，增强亲和力。


<!-- session-2026-04-29-superset-completion -->
## 2.5 看 Capstone Skill 是怎么搭的

前面是"加载 + 跑"——把 capstone_assistant skill 当黑盒用。这里揭示 skill **内部 pipeline.py 的核心逻辑**，让你知道怎么自己造一个企业级 skill。

核心架构：**4 大组件 + 1 个 router**

```
用户 query
   ↓
[Router Planner Prompt] ── 决策：调哪个组件？
   ↓
   ├── [Multi-Agent] ←─ 复杂任务（设计 / 写代码）
   ├── [MCP Tools]   ←─ 数据查询（订单 / 员工 / 文档）
   ├── [Agentic RAG] ←─ 知识库问答
   └── [LLM Direct]  ←─ 简单对话
   ↓
[Observability @observe] ── 全程 trace
   ↓
最终回答
```


In [ ]:
# 摘录 Capstone Skill 的核心 router 逻辑（来自 capstone_assistant/pipeline.py）

CAPSTONE_ROUTER_PROMPT = """你是路由 Planner。根据用户 query 选择最合适的组件：

可用组件：
- multi_agent: 复杂多步任务，需要多个专家协作（如"设计一个 X，写代码再 review"）
- mcp_tools: 数据查询任务（订单查询、员工信息、文档检索）
- agentic_rag: 企业知识库问答（产品文档、政策手册）
- llm_direct: 简单对话或不需要外部信息的回答

输出 JSON：{
  "component": "multi_agent" | "mcp_tools" | "agentic_rag" | "llm_direct",
  "reason": "<为什么选这个>",
  "args": <组件特定参数>
}
"""

# 简化版 router 实现（生产里这一步是真 LLM 调用）
def capstone_route(user_query: str) -> dict:
    """根据 query 关键词决定走哪个组件。生产里 LLM 输出 JSON。"""
    q = user_query.lower()
    if any(k in q for k in ["设计", "代码", "实现", "review"]):
        return {"component": "multi_agent", "reason": "多步设计 + 实现 + 评审任务"}
    elif any(k in q for k in ["订单", "员工", "查询", "od2", "e0"]):
        return {"component": "mcp_tools", "reason": "结构化数据查询"}
    elif any(k in q for k in ["政策", "文档", "手册", "规定", "knowledge"]):
        return {"component": "agentic_rag", "reason": "知识库问答"}
    else:
        return {"component": "llm_direct", "reason": "简单对话"}

# Demo: 看不同 query 路由到哪个组件
print("=" * 78)
print("        Capstone Skill 内部 router 逻辑 demo")
print("=" * 78)
demos = [
    "帮我设计一个用户认证模块，写出 Python 代码并 review",
    "查一下订单 OD2024 的状态",
    "公司的退款政策是怎么样的？",
    "你好",
]
for q in demos:
    decision = capstone_route(q)
    print(f"\n>>> Query: {q}")
    print(f"    → component: {decision['component']}")
    print(f"    → reason: {decision['reason']}")

print("\n" + "=" * 78)
print("\n这是 Capstone Skill 的核心——把 Day 4-5 的所有组件统一在一个 router 后面。")
print("每个组件内部的实现见 capstone_assistant/pipeline.py 完整版。")
print("\n@observe 装饰器把整个 router 流程包起来——每次 capstone_route 调用")
print("都会在 observability backend（Mock / Langfuse）里记录一条 span。")


## 3. 看完整 trace tree

理论上这一格应该打印出每个 query 的完整 span 树（Planner → RAG/MCP → Reviewer 三层嵌套）。**实际上你会看到 trace tree 是空的**——这是当前 Capstone 的一个已知缺口。

原因：`pipeline.py` 内部没有用 `@observe` 装饰它的子调用，所以 `MockObserver` 没收到 span 事件。修这个问题需要在 5 days 那边的 `pipeline.py` 加 6-7 行 `@observe` 装饰器——后续 PR 会补上，这一节先把缺口标出来，让你知道"trace 集成是工程整合的最后一公里"，不是写完代码就自动有的。

In [4]:
from utils.observability import observer

print("过去 4 个 query 的完整 trace tree:")
observer.print_tree()
import json
print("\nSummary:")
print(json.dumps(observer.summary(), indent=2, ensure_ascii=False))


过去 4 个 query 的完整 trace tree:

Summary:
{
  "n_traces": 0,
  "n_total_spans": 0,
  "total_duration_ms": 0,
  "tokens": {
    "prompt": 0,
    "completion": 0,
    "total": 0
  },
  "cost_usd": 0.0
}


## 4. Skill 包装的工程价值

Capstone 整体被打包成 Anthropic Skill (`capstone_assistant/`)：

```
capstone_assistant/
├── SKILL.md             # YAML frontmatter (name + description) + workflow body
├── pipeline.py          # upgraded_pipeline() 主入口
├── eval.py              # batch_eval() 评测框架
└── reference/
    ├── architecture.md  # 详细架构（按需载入，progressive disclosure）
    └── eval_cases.jsonl # 10 个评测用例
```

为什么这个文件夹格式重要？

工程实践里，"复用一套 Agent 系统"的传统办法有几种：复制粘贴代码（容易漂移）、做成 PyPI 包（重，且失去 prompt 这种文本资产）、写成微服务（更重，跨团队部署成本高）。Skill 格式介于这几者之间——**把代码、prompt、评测、文档放在同一个文件夹里**，团队 git clone 就能用、改一行就能定制。

更重要的是**它有 LLM 友好的元信息**（SKILL.md 里的 description 字段）。Claude / Cursor 这类客户端读到 description 后能自动决定何时调用这个 Skill——这就是为什么你不需要手动注册 Skill 入口，LLM 自己会找。

## 5. 跑评测看综合表现

batch_eval 默认跑主 repo 的 `data/eval_zh_extended.jsonl`（120 题），统计 per-category 准确率。这是产品上线前的标准动作——**不要用感觉判断 agent 系统好不好用，要用评测集**。

扩展集比 10 题 smoke test 更接近真实评测：同一能力会有多种问法，也包含 OOD 和中文常识 sanity check。题量变大后准确率不必追求 90% 以上；稳定落在 60% 以上，且失败样例可解释，才更适合作为迭代基线。

In [5]:
import json
from pathlib import Path

# 同样用 _root（前面 cell 已计算好），从任意 CWD 跑
_capstone_dir = _root / "assets" / "enterprise_5days" / "skills_demo" / "capstone_assistant"
sys.path.insert(0, str(_capstone_dir))
import eval as capstone_eval

# 默认使用主 repo 扩展评测集；课堂快跑可改回 Skill 自带 reference/eval_cases.jsonl
_eval_cases = _root / "data" / "eval_zh_extended.jsonl"
result = capstone_eval.batch_eval(str(_eval_cases))
print(f"Capstone 整体准确率: {result['success_rate']:.0%}")
print("\n按类别:")
for cat, acc in result["per_category"].items():
    print(f"  {cat:<10} {acc:.0%}")

print(f"\n失败 case 示例:")
fails = [r for r in result["details"] if not r["ok"]][:3]
for f in fails:
    print(f"  Q: {f['query']}")
    print(f"    A: {f['answer'][:80]}")


[OK] 已加载配置: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\.env
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus
  Embedding: dashscope / text-embedding-v3
[LLM] dashscope / qwen-plus
[Embedding] dashscope / text-embedding-v3 (dim=1024)


✓ 已添加 7 个文档，总计 7 个


Capstone 整体准确率: 90%

按类别:
  hr         100%
  tech       100%
  product    100%
  mcp        100%
  ood        0%
  direct     100%

失败 case 示例:
  Q: 公司有医疗保险吗？
    A: 信息不足


In [ ]:
# 自检：5 天合体的 Capstone 全栈是否跑通
def verify_app8() -> bool:
    print("=" * 56)
    print("自检 · App8 Production Capstone")
    print("=" * 56)
    checks: list[tuple[str, bool, str]] = []
    g = globals()

    # 1. capstone Skill 模块加载（§1 一键导入 cell）
    if "capstone" in g and hasattr(g["capstone"], "upgraded_pipeline"):
        checks.append(("Capstone Skill 模块加载（capstone.upgraded_pipeline 可调）",
                       True, "ok"))
    else:
        checks.append(("Capstone Skill 模块加载", False, "⏭ §1 import cell 未跑"))

    # 2. batch_eval 准确率 ≥ 60%（§5 跑出 result）
    res = g.get("result")
    if isinstance(res, dict) and "success_rate" in res:
        acc = res["success_rate"]
        checks.append(("Capstone batch eval 准确率 ≥ 60%",
                       acc >= 0.6, f"{acc:.0%}"))
    else:
        checks.append(("Capstone batch eval 准确率 ≥ 60%", False,
                       "⏭ §5 eval cell 未跑或 result schema 异常"))

    # 3. 评测覆盖 ≥4 个类别
    per = res.get("per_category") if isinstance(res, dict) else None
    if isinstance(per, dict):
        cats = list(per.keys())
        checks.append(("评测覆盖 ≥4 个类别",
                       len(cats) >= 4, f"{cats}"))
    else:
        checks.append(("评测覆盖 ≥4 个类别", False, "⏭ result 无 per_category"))

    # 4. knowledge + mcp 两条核心路径都成功
    if isinstance(per, dict):
        knowledge_cats = ["hr", "tech", "product"]
        knowledge_ok = any(per.get(cat, 0) > 0 for cat in knowledge_cats)
        mcp_ok = per.get("mcp", 0) > 0
        ok = knowledge_ok and mcp_ok
        knowledge_detail = ", ".join(f"{cat}={per.get(cat, 0):.0%}" for cat in knowledge_cats)
        checks.append(("知识库路径 + mcp 工具路径都成功",
                       ok, f"{knowledge_detail}, mcp={per.get('mcp', 0):.0%}"))
    else:
        checks.append(("知识库路径 + mcp 工具路径都成功", False, "⏭ result 无 per_category"))

    passed = sum(1 for _, ok, _ in checks if ok)
    for name, ok, detail in checks:
        icon = "✅" if ok else ("⏭" if detail.startswith("⏭") else "❌")
        print(f"  {icon} {name}  ({detail})")
    print(f"\n通过 {passed}/{len(checks)}")
    if passed == len(checks):
        print("5 天工业栈合体已跑通。可 fork capstone_assistant Skill 改造成自家业务。")
    elif passed >= 2:
        print("部分通过——核心路径已验证；剩余项请检查 §5 eval cell 输出。")
    else:
        print("未通过——请检查 LLM API Key + 网络后重跑。")
    return passed == len(checks)


verify_app8()


## 收尾：5 天的合体清单

回头看，Capstone 把前面学到的能力按这个表格合体：

| 组件 | 来自 | App 入口 |
|---|---|---|
| Multi-Agent (Planner/Worker/Reviewer) | 5 天版 Day 4 上午 | App4 |
| MCP 工具 (订单/库存) | 5 天版 Day 4 下午 | App5 |
| Agentic RAG (Hybrid+Self) | 5 天版 Day 5 上午 | App2 |
| LLMOps Trace | 5 天版 Day 5 下午 | App7 |
| Skill 打包 | 5 天版 Day 5 下午 | App6 |

**这套五件合体是 2026 工业级 Agent 系统的最小可行组合**——比这少就缺关键能力，比这多就是过度设计的开始。

把这套 fork 成你公司业务的下一步：

1. 改 `pipeline.py` 里的路由逻辑，把 4 类 query 换成你业务的 query 类型
2. 改 MCP server 暴露你公司的真实 API（订单、库存、CRM、ERP）
3. 改 RAG 知识库换成你公司的内部文档
4. 改 `eval_cases.jsonl` 写你业务关心的评测用例
5. 接 Langfuse / 自家观测平台拿到真 dashboard

到这里你已经走完 5 天工业栈的全部内容。后面要做的不是再学新模式，而是在你具体的业务上把这套跑稳——加监控、加权限、加评测、迭代提示词。这些都不在课程范围内，因为它们高度依赖你公司的具体场景。